# اليوم الثالث — التعلم الخاضع للإشراف
نبدأ بنموذج التصنيف الذي بُني في اليوم الثاني، ثم نقرأ العتبة، ونبني الانحدار، ونقارن نموذجين.

## المحطة 0 — تحميل البيانات

### 0.1 استيراد المكتبات الأساسية

In [ ]:
import numpy as np
import pandas as pd

### 0.2 قراءة الملف

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'
df = pd.read_csv(DATA_URL)
print('تم التحميل ✅', df.shape)

# المحطة 1 — بناء مسار التصنيف

### 1.1 تحديد الهدف

In [ ]:
y = df['churned_30d'].copy()

### 1.2 تحديد الأعمدة المستبعدة

In [ ]:
drop_cols = [
    'customer_id', 'churned_30d',
    'refund_issued', 'support_ticket_after_snapshot', 'next_month_orders',
    'signup_date', 'snapshot_date'
]

### 1.3 إنشاء X

In [ ]:
X = df.drop(columns=drop_cols, errors="ignore")

### 1.4 فصل الاختبار النهائي

In [ ]:
from sklearn.model_selection import train_test_split

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

### 1.5 فصل التدريب والتحقق

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42
)

### 1.6 تحديد أنواع الأعمدة

In [ ]:
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

### 1.7 تجهيز الأعمدة الرقمية

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

number_steps = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

### 1.8 تجهيز الأعمدة الفئوية

In [ ]:
from sklearn.preprocessing import OneHotEncoder

category_steps = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])

### 1.9 جمع المعالجة

In [ ]:
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer([
    ('numbers', number_steps, numeric_cols),
    ('categories', category_steps, categorical_cols)
])

### 1.10 إنشاء Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000))
])

### 1.11 تدريب النموذج

In [ ]:
logistic.fit(X_train, y_train)

### 1.12 استخراج احتمالات التحقق

In [ ]:
val_probability = logistic.predict_proba(X_val)[:, 1]

# المحطة 2 — أثر العتبة

### 2.1 تحويل الاحتمال إلى فئة عند عتبة 0.50

In [ ]:
prediction_050 = (val_probability >= 0.50).astype(int)

### 2.2 حساب مصفوفة الالتباس

In [ ]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_val, prediction_050).ravel()
print('TP=', tp, 'FP=', fp, 'FN=', fn, 'TN=', tn)

### 2.3 حساب Precision وRecall

In [ ]:
from sklearn.metrics import precision_score, recall_score

print('Precision:', round(precision_score(y_val, prediction_050), 3))
print('Recall:', round(recall_score(y_val, prediction_050), 3))

### 2.4 مقارنة ثلاث عتبات
الحلقة تكرر القياسات نفسها عند ثلاث عتبات فقط.

In [ ]:
threshold_rows = []
for threshold in [0.30, 0.50, 0.70]:
    prediction = (val_probability >= threshold).astype(int)
    threshold_rows.append({
        'العتبة': threshold,
        'Precision': precision_score(y_val, prediction, zero_division=0),
        'Recall': recall_score(y_val, prediction, zero_division=0)
    })

### 2.5 عرض المقارنة

In [ ]:
display(pd.DataFrame(threshold_rows).round(3))

# المحطة 3 — بناء نموذج انحدار

### 3.1 تحديد هدف الانحدار

In [ ]:
y_reg = df['avg_basket_sar'].copy()

### 3.2 إنشاء خصائص الانحدار

In [ ]:
reg_drop = drop_cols + ['avg_basket_sar']
X_reg = df.drop(columns=reg_drop, errors='ignore')

### 3.3 تقسيم بيانات الانحدار

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

### 3.4 تحديد أنواع أعمدة الانحدار

In [ ]:
reg_numeric = X_reg.select_dtypes(include='number').columns.tolist()
reg_categorical = X_reg.select_dtypes(exclude='number').columns.tolist()

### 3.5 إنشاء معالجة الانحدار

In [ ]:
reg_preprocess = ColumnTransformer([
    ('numbers', number_steps, reg_numeric),
    ('categories', category_steps, reg_categorical)
])

### 3.6 إنشاء Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = Pipeline([
    ('preprocess', reg_preprocess),
    ('model', LinearRegression())
])

### 3.7 تدريب الانحدار

In [ ]:
linear_model.fit(Xr_train, yr_train)

### 3.8 إنتاج توقعات الانحدار

In [ ]:
reg_prediction = linear_model.predict(Xr_test)

### 3.9 حساب MAE

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(yr_test, reg_prediction)
print('MAE:', round(mae, 2), 'ريال')

### 3.10 حساب RMSE

In [ ]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(yr_test, reg_prediction)
print('RMSE:', round(rmse, 2), 'ريال')

### 3.11 حساب R²

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(yr_test, reg_prediction)
print('R²:', round(r2, 3))

# المحطة 4 — مقارنة Logistic Regression وRandom Forest

### 4.1 حساب PR-AUC للنموذج اللوجستي

In [ ]:
from sklearn.metrics import average_precision_score

logistic_pr_auc = average_precision_score(y_val, val_probability)
print('Logistic PR-AUC:', round(logistic_pr_auc, 3))

### 4.2 إنشاء Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestClassifier(
        n_estimators=100, max_depth=8, min_samples_leaf=5, random_state=42
    ))
])

### 4.3 تدريب Random Forest

In [ ]:
forest.fit(X_train, y_train)

### 4.4 استخراج احتمالات Random Forest

In [ ]:
forest_probability = forest.predict_proba(X_val)[:, 1]

### 4.5 حساب PR-AUC للغابة

In [ ]:
forest_pr_auc = average_precision_score(y_val, forest_probability)
print('Random Forest PR-AUC:', round(forest_pr_auc, 3))

### 4.6 عرض مقارنة واضحة

In [ ]:
comparison = pd.DataFrame({
    'النموذج': ['Logistic Regression', 'Random Forest'],
    'Validation PR-AUC': [logistic_pr_auc, forest_pr_auc]
})
display(comparison.round(3))

# المحطة 5 — فتح الاختبار مرة واحدة

### 5.1 اختيار النموذج الأفضل على التحقق

In [ ]:
chosen_model = forest if forest_pr_auc > logistic_pr_auc else logistic
chosen_name = 'Random Forest' if forest_pr_auc > logistic_pr_auc else 'Logistic Regression'
print('النموذج المختار:', chosen_name)

### 5.2 قياسه على الاختبار النهائي

In [ ]:
test_probability = chosen_model.predict_proba(X_test)[:, 1]
test_pr_auc = average_precision_score(y_test, test_probability)
print('Final Test PR-AUC:', round(test_pr_auc, 3))

## الخلاصة
فصلنا كل قرار: البيانات، التقسيم، المعالجة، التدريب، الاحتمال، العتبة، المقاييس، ثم المقارنة والاختبار النهائي.